# Python OOP & Professional Features
### Reference Notebook — Library Management System Context
Every concept is demonstrated through a Library domain so everything connects.

**Topics covered:**
- Classes, Objects, Constructors
- Encapsulation, Property Decorators
- Inheritance, MRO
- Polymorphism, Abstraction, ABC
- Magic Methods, Operator Overloading
- Mixins
- Exception Handling, Custom Exceptions
- Iterators, Generators
- Decorators
- Context Managers

---
## 1. Classes, Objects & Constructors

In [ ]:
# A class is a blueprint. An object is an instance built from that blueprint.
# __init__ is the constructor — it runs automatically when you create an object.

class Book:
    # Class variable — shared across ALL Book objects
    total_books = 0

    def __init__(self, title, author, isbn):
        # Instance variables — unique to each object
        # 'self' refers to the specific object being created
        self.title = title
        self.author = author
        self.isbn = isbn
                  # default value set at creation
        Book.total_books += 1             # increment class variable

    def checkout(self):
        """Mark book as checked out."""
        self.is_available = False
        print(f"'{self.title}' has been checked out.")

    def return_book(self):
        """Mark book as returned."""
        self.is_available = True
        print(f"'{self.title}' has been returned.")


# Creating objects (instances)
book1 = Book("Clean Code", "Robert Martin", "978-01")
book2 = Book("The Pragmatic Programmer", "Andrew Hunt", "978-02")

print(book1.title)              # accessing instance variable
print(Book.total_books)         # accessing class variable — 2
book1.checkout()                # calling a method
print(book1.is_available)       # False
print(book2.is_available)       # True — each object has its own state

Clean Code
2
'Clean Code' has been checked out.
False
True


---
## 2. Encapsulation

In [2]:
# Encapsulation = bundling data + methods, and restricting direct access to internals.
# Convention in Python:
#   self.name     → public   (anyone can access)
#   self._name    → protected (internal use, but accessible — just a warning)
#   self.__name   → private  (name-mangled, not directly accessible outside class)

class Member:
    def __init__(self, name, member_id):
        self.name = name                   # public
        self._member_id = member_id        # protected — internal, still accessible
        self.__fine_amount = 0.0           # private — hidden from outside
        self.borrowed_books = []

    def add_fine(self, amount):
        """Only way to modify fine — controlled access."""
        if amount < 0:
            raise ValueError("Fine cannot be negative")
        self.__fine_amount += amount

    def get_fine(self):
        """Read-only access to fine."""
        return self.__fine_amount

    def clear_fine(self):
        self.__fine_amount = 0.0
        print(f"{self.name}'s fine cleared.")


member = Member("Hasnain", "M001")
member.add_fine(150)
print(member.get_fine())           # 150.0 — works through method

# print(member.__fine_amount)      # AttributeError — private, blocked
# Python name-mangles it to _Member__fine_amount internally
print(member._Member__fine_amount) # 150.0 — accessible but you shouldn't do this

print(member._member_id)           # M001 — protected, accessible but discouraged

150.0
150.0
M001


---
## 3. Property Decorators

In [3]:
# @property lets you access a method like an attribute.
# Cleaner encapsulation — add validation without changing the public interface.

class Book:
    def __init__(self, title, author, pages):
        self.title = title
        self.__author = author
        self.__pages = pages

    @property
    def author(self):
        """Getter — called when you read book.author"""
        return self.__author

    @author.setter
    def author(self, value):
        """Setter — called when you do book.author = 'something'"""
        if not isinstance(value, str) or len(value.strip()) == 0:
            raise ValueError("Author must be a non-empty string")
        self.__author = value.strip()

    @property
    def pages(self):
        return self.__pages

    @pages.setter
    def pages(self, value):
        if not isinstance(value, int) or value <= 0:
            raise ValueError("Pages must be a positive integer")
        self.__pages = value

    @property
    def is_long_read(self):
        """Computed property — no setter needed, derived from data."""
        return self.__pages > 400


book = Book("Clean Code", "Robert Martin", 431)

print(book.author)           # Robert Martin  — calls getter, looks like attribute
book.author = "Bob Martin"   # calls setter with validation
print(book.author)           # Bob Martin
print(book.is_long_read)     # True — computed property

try:
    book.pages = -5          # triggers setter validation
except ValueError as e:
    print(f"Caught: {e}")

Robert Martin
Bob Martin
True
Caught: Pages must be a positive integer


---
## 4. Inheritance & super()

In [4]:
# Inheritance lets a child class reuse everything from a parent class.
# super() calls the parent's method — essential in __init__ chains.

class LibraryItem:
    """Parent class — common features all library items share."""
    def __init__(self, title, item_id):
        self.title = title
        self.item_id = item_id
        self.is_available = True

    def checkout(self):
        self.is_available = False
        print(f"'{self.title}' checked out.")

    def return_item(self):
        self.is_available = True
        print(f"'{self.title}' returned.")

    def status(self):
        state = "Available" if self.is_available else "Checked Out"
        return f"[{self.item_id}] {self.title} — {state}"


class Book(LibraryItem):   # Book inherits from LibraryItem
    def __init__(self, title, item_id, author, isbn):
        super().__init__(title, item_id)   # call parent __init__ first
        self.author = author               # then add Book-specific data
        self.isbn = isbn


class Magazine(LibraryItem):
    def __init__(self, title, item_id, issue_number, publisher):
        super().__init__(title, item_id)
        self.issue_number = issue_number
        self.publisher = publisher


class DVD(LibraryItem):
    def __init__(self, title, item_id, director, duration_mins):
        super().__init__(title, item_id)
        self.director = director
        self.duration_mins = duration_mins


book = Book("Clean Code", "B001", "Robert Martin", "978-01")
mag  = Magazine("Time", "M001", 45, "Time USA LLC")
dvd  = DVD("Interstellar", "D001", "Christopher Nolan", 169)

# All inherited from LibraryItem
book.checkout()
print(book.status())
print(mag.status())

# isinstance checks
print(isinstance(book, Book))          # True
print(isinstance(book, LibraryItem))   # True — Book IS a LibraryItem
print(isinstance(book, Magazine))      # False

'Clean Code' checked out.
[B001] Clean Code — Checked Out
[M001] Time — Available
True
True
False


---
## 5. MRO — Method Resolution Order

In [5]:
# When multiple inheritance is involved, Python uses MRO (C3 Linearization)
# to decide WHICH parent's method to call.
# Rule: left to right, depth first, no class repeated.

class LibraryItem:
    def describe(self):
        return "I am a LibraryItem"

class Borrowable:
    def describe(self):
        return "I can be borrowed"

class Reservable:
    def describe(self):
        return "I can be reserved"

class Book(LibraryItem, Borrowable, Reservable):
    pass   # no override — will use parent method


book = Book()
print(book.describe())     # "I am a LibraryItem" — leftmost parent wins

# See the full resolution order
print(Book.__mro__)
# (Book, LibraryItem, Borrowable, Reservable, object)

# ---- super() chains through MRO ----
class A:
    def greet(self):
        print("A")

class B(A):
    def greet(self):
        super().greet()    # calls A.greet
        print("B")

class C(A):
    def greet(self):
        super().greet()    # calls A.greet
        print("C")

class D(B, C):             # MRO: D → B → C → A
    def greet(self):
        super().greet()    # chains: B → C → A
        print("D")

D().greet()   # prints: A, C, B, D  (MRO order from bottom up)

I am a LibraryItem
(<class '__main__.Book'>, <class '__main__.LibraryItem'>, <class '__main__.Borrowable'>, <class '__main__.Reservable'>, <class 'object'>)
A
C
B
D


---
## 6. Polymorphism

In [6]:
# Polymorphism = same interface, different behaviour per class.
# You call the same method name on different objects — each responds its own way.

class LibraryItem:
    def __init__(self, title, item_id):
        self.title = title
        self.item_id = item_id

    def get_info(self):
        raise NotImplementedError("Subclass must implement get_info")

    def get_late_fee(self, days_overdue):
        raise NotImplementedError("Subclass must implement get_late_fee")


class Book(LibraryItem):
    def __init__(self, title, item_id, author):
        super().__init__(title, item_id)
        self.author = author

    def get_info(self):                            # Book's version
        return f"📖 Book  | {self.item_id} | {self.title} by {self.author}"

    def get_late_fee(self, days_overdue):          # Rs. 10/day for books
        return days_overdue * 10


class Magazine(LibraryItem):
    def __init__(self, title, item_id, issue):
        super().__init__(title, item_id)
        self.issue = issue

    def get_info(self):                            # Magazine's version
        return f"📰 Mag   | {self.item_id} | {self.title} Issue #{self.issue}"

    def get_late_fee(self, days_overdue):          # Rs. 5/day for magazines
        return days_overdue * 5


class DVD(LibraryItem):
    def __init__(self, title, item_id, director):
        super().__init__(title, item_id)
        self.director = director

    def get_info(self):                            # DVD's version
        return f"🎬 DVD   | {self.item_id} | {self.title} dir. {self.director}"

    def get_late_fee(self, days_overdue):          # Rs. 20/day for DVDs
        return days_overdue * 20


# Polymorphism in action — one loop, three different behaviours
items = [
    Book("Clean Code", "B001", "Robert Martin"),
    Magazine("Time", "M001", 45),
    DVD("Interstellar", "D001", "Nolan")
]

print("=== Library Catalogue ===")
for item in items:
    print(item.get_info())           # correct version called per type

print("\n=== Late Fees (5 days overdue) ===")
for item in items:
    print(f"{item.title}: Rs. {item.get_late_fee(5)}")

=== Library Catalogue ===
📖 Book  | B001 | Clean Code by Robert Martin
📰 Mag   | M001 | Time Issue #45
🎬 DVD   | D001 | Interstellar dir. Nolan

=== Late Fees (5 days overdue) ===
Clean Code: Rs. 50
Time: Rs. 25
Interstellar: Rs. 100


---
## 7. Abstraction & ABC

In [7]:
# ABC (Abstract Base Class) enforces a contract.
# Any class inheriting from an ABC MUST implement all @abstractmethod methods.
# You cannot instantiate an ABC directly.

from abc import ABC, abstractmethod

class LibraryItem(ABC):              # ABC = abstract base class
    def __init__(self, title, item_id):
        self.title = title
        self.item_id = item_id
        self.is_available = True

    @abstractmethod
    def get_info(self) -> str:
        """Every item type MUST implement this."""
        pass

    @abstractmethod
    def get_late_fee(self, days: int) -> float:
        """Every item type MUST define its own fee structure."""
        pass

    # Concrete method — shared logic, no override needed
    def checkout(self):
        self.is_available = False

    def return_item(self):
        self.is_available = True


class Book(LibraryItem):
    def __init__(self, title, item_id, author):
        super().__init__(title, item_id)
        self.author = author

    def get_info(self) -> str:        # MUST implement — abstract contract
        return f"Book: {self.title} by {self.author}"

    def get_late_fee(self, days: int) -> float:
        return days * 10.0


# Trying to skip an abstract method
class BadItem(LibraryItem):
    def get_info(self):
        return "info"
    # Missing get_late_fee — Python will block instantiation


book = Book("Clean Code", "B001", "Martin")   # works
print(book.get_info())
print(book.get_late_fee(3))    # 30.0

try:
    item = LibraryItem("X", "001")   # TypeError — can't instantiate ABC
except TypeError as e:
    print(f"Caught: {e}")

try:
    bad = BadItem("X", "002")        # TypeError — missing abstract method
except TypeError as e:
    print(f"Caught: {e}")

Book: Clean Code by Martin
30.0
Caught: Can't instantiate abstract class LibraryItem without an implementation for abstract methods 'get_info', 'get_late_fee'
Caught: Can't instantiate abstract class BadItem without an implementation for abstract method 'get_late_fee'


---
## 8. Magic Methods (Dunder Methods)

In [8]:
# Magic methods define how your objects behave with Python built-ins.
# __str__, __repr__, __len__, __eq__, __lt__, __contains__, __bool__

class Book:
    def __init__(self, title, author, isbn, pages):
        self.title = title
        self.author = author
        self.isbn = isbn
        self.pages = pages

    def __str__(self):
        """Human-readable string — used by print()"""
        return f"{self.title} by {self.author}"

    def __repr__(self):
        """Developer-readable — used in REPL, lists, debugger"""
        return f"Book(title={self.title!r}, isbn={self.isbn!r})"

    def __len__(self):
        """len(book) → number of pages"""
        return self.pages

    def __eq__(self, other):
        """book1 == book2 → True if same ISBN"""
        if not isinstance(other, Book):
            return NotImplemented
        return self.isbn == other.isbn

    def __lt__(self, other):
        """book1 < book2 → compare by pages (enables sorting)"""
        return self.pages < other.pages

    def __bool__(self):
        """bool(book) → True if book has a title"""
        return bool(self.title)


b1 = Book("Clean Code", "Martin", "978-01", 431)
b2 = Book("Clean Code", "Martin", "978-01", 431)   # same ISBN
b3 = Book("SICP", "Abelson", "978-03", 657)

print(b1)                  # Clean Code by Martin          — __str__
print(repr(b1))            # Book(title='Clean Code', ...)  — __repr__
print(len(b1))             # 431                            — __len__
print(b1 == b2)            # True (same ISBN)               — __eq__
print(b1 == b3)            # False
print(b1 < b3)             # True (431 < 657)               — __lt__
print(bool(b1))            # True                           — __bool__

books = [b3, b1]
print(sorted(books))       # sorted by pages — works because __lt__ defined
print([b1, b3])            # uses __repr__ for display

Clean Code by Martin
Book(title='Clean Code', isbn='978-01')
431
True
False
True
True
[Book(title='Clean Code', isbn='978-01'), Book(title='SICP', isbn='978-03')]
[Book(title='Clean Code', isbn='978-01'), Book(title='SICP', isbn='978-03')]


---
## 9. Operator Overloading

In [9]:
# Operator overloading = defining what +, -, *, in, etc. mean for YOUR objects.
# These are all magic methods under the hood.

class Library:
    def __init__(self, name):
        self.name = name
        self.items = []

    def __add__(self, item):
        """library + book → adds book to collection"""
        self.items.append(item)
        return self                  # return self to allow chaining

    def __sub__(self, item):
        """library - book → removes book from collection"""
        if item in self.items:
            self.items.remove(item)
        return self

    def __len__(self):
        """len(library) → total items"""
        return len(self.items)

    def __contains__(self, item):
        """book in library → True/False"""
        return item in self.items

    def __str__(self):
        return f"Library '{self.name}' — {len(self)} items"

    def __getitem__(self, index):
        """library[0] → first item"""
        return self.items[index]


class Book:
    def __init__(self, title, isbn):
        self.title = title
        self.isbn = isbn

    def __eq__(self, other):
        return isinstance(other, Book) and self.isbn == other.isbn

    def __repr__(self):
        return f"Book({self.title!r})"


lib = Library("City Library")
b1 = Book("Clean Code", "978-01")
b2 = Book("SICP", "978-02")
b3 = Book("Pragmatic Programmer", "978-03")

lib + b1 + b2 + b3           # chained + operator
print(lib)                   # Library 'City Library' — 3 items
print(len(lib))              # 3
print(b1 in lib)             # True
print(lib[0])                # Book('Clean Code')

lib - b2                     # remove SICP
print(len(lib))              # 2
print(b2 in lib)             # False

Library 'City Library' — 3 items
3
True
Book('Clean Code')
2
False


---
## 10. Mixins

In [10]:
# Mixins are small, focused classes that add ONE capability.
# They are not meant to be used alone — they 'mix in' to other classes.
# Solve the problem of sharing behaviour without deep inheritance.

from datetime import datetime

class LogMixin:
    """Adds logging capability to any class."""
    def log(self, action: str):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[LOG {timestamp}] {action}")


class SearchMixin:
    """Adds search capability — assumes self.items exists."""
    def search_by_title(self, keyword: str):
        keyword = keyword.lower()
        return [item for item in self.items if keyword in item.title.lower()]

    def search_by_id(self, item_id: str):
        return next((item for item in self.items if item.item_id == item_id), None)


class ExportMixin:
    """Adds export capability — assumes self.items exists."""
    def export_titles(self) -> list:
        return [item.title for item in self.items]


# Library gets ALL THREE capabilities through mixins
class Library(LogMixin, SearchMixin, ExportMixin):
    def __init__(self, name):
        self.name = name
        self.items = []

    def add_item(self, item):
        self.items.append(item)
        self.log(f"Added: '{item.title}'")   # from LogMixin

    def remove_item(self, item_id):
        item = self.search_by_id(item_id)     # from SearchMixin
        if item:
            self.items.remove(item)
            self.log(f"Removed: '{item.title}'")  # from LogMixin


class Book:
    def __init__(self, title, item_id):
        self.title = title
        self.item_id = item_id


lib = Library("City Library")
lib.add_item(Book("Clean Code", "B001"))
lib.add_item(Book("Clean Architecture", "B002"))
lib.add_item(Book("SICP", "B003"))

results = lib.search_by_title("clean")    # from SearchMixin
print([b.title for b in results])         # ['Clean Code', 'Clean Architecture']

print(lib.export_titles())                # from ExportMixin

lib.remove_item("B002")

[LOG 2026-06-24 11:21:37] Added: 'Clean Code'
[LOG 2026-06-24 11:21:37] Added: 'Clean Architecture'
[LOG 2026-06-24 11:21:37] Added: 'SICP'
['Clean Code', 'Clean Architecture']
['Clean Code', 'Clean Architecture', 'SICP']
[LOG 2026-06-24 11:21:37] Removed: 'Clean Architecture'


---
## 11. Custom Exceptions & Exception Handling

In [11]:
# Custom exceptions make errors meaningful and catchable by type.
# Build a hierarchy: one base, specific children.

# --- Exception Hierarchy ---
class LibraryError(Exception):
    """Base exception for all library errors."""
    pass

class ItemNotFoundError(LibraryError):
    def __init__(self, item_id):
        super().__init__(f"Item with ID '{item_id}' not found in library")
        self.item_id = item_id

class ItemNotAvailableError(LibraryError):
    def __init__(self, title):
        super().__init__(f"'{title}' is currently checked out")
        self.title = title

class MemberLimitError(LibraryError):
    MAX_BOOKS = 3
    def __init__(self, member_name):
        super().__init__(
            f"{member_name} has reached the {self.MAX_BOOKS}-book borrowing limit"
        )

class DuplicateItemError(LibraryError):
    def __init__(self, isbn):
        super().__init__(f"Item with ISBN '{isbn}' already exists")


# --- Using exceptions in real logic ---
class Book:
    def __init__(self, title, item_id, isbn):
        self.title = title
        self.item_id = item_id
        self.isbn = isbn
        self.is_available = True

class Member:
    def __init__(self, name):
        self.name = name
        self.borrowed = []

class Library:
    def __init__(self):
        self.items = {}

    def add_book(self, book: Book):
        if book.isbn in self.items:
            raise DuplicateItemError(book.isbn)  # raise specific exception
        self.items[book.isbn] = book

    def borrow(self, isbn: str, member: Member):
        if isbn not in self.items:
            raise ItemNotFoundError(isbn)

        book = self.items[isbn]

        if not book.is_available:
            raise ItemNotAvailableError(book.title)

        if len(member.borrowed) >= 3:
            raise MemberLimitError(member.name)

        book.is_available = False
        member.borrowed.append(book)
        print(f"{member.name} borrowed '{book.title}'")


# --- Exception handling in action ---
lib = Library()
lib.add_book(Book("Clean Code", "B001", "978-01"))

hasnain = Member("Hasnain")

try:
    lib.borrow("978-01", hasnain)       # succeeds
    lib.borrow("978-01", hasnain)       # fails — not available
except ItemNotAvailableError as e:
    print(f"Not available: {e}")
except ItemNotFoundError as e:
    print(f"Not found: {e}")
except LibraryError as e:
    print(f"Library error: {e}")        # catches ANY library error
finally:
    print("Borrow attempt complete.")   # always runs


# Duplicate check
try:
    lib.add_book(Book("Clean Code", "B001", "978-01"))   # same ISBN
except DuplicateItemError as e:
    print(f"Duplicate: {e}")

Hasnain borrowed 'Clean Code'
Not available: 'Clean Code' is currently checked out
Borrow attempt complete.
Duplicate: Item with ISBN '978-01' already exists


---
## 12. Iterators

In [12]:
# An iterator is an object that produces values one at a time.
# Requires two magic methods:
#   __iter__  → returns the iterator object (usually self)
#   __next__  → returns the next value, raises StopIteration when done

class Book:
    def __init__(self, title, item_id):
        self.title = title
        self.item_id = item_id
        self.is_available = True

    def __repr__(self):
        return f"Book({self.title!r})"


class LibraryCatalogue:
    """Makes the library itself iterable — you can loop over it directly."""
    def __init__(self):
        self.items = []

    def add(self, item):
        self.items.append(item)

    def __iter__(self):
        """Called when iteration starts (e.g. for item in library)."""
        self._index = 0
        return self

    def __next__(self):
        """Called on each iteration step."""
        if self._index >= len(self.items):
            raise StopIteration          # signals end of iteration
        item = self.items[self._index]
        self._index += 1
        return item

    def __len__(self):
        return len(self.items)


catalogue = LibraryCatalogue()
catalogue.add(Book("Clean Code", "B001"))
catalogue.add(Book("SICP", "B002"))
catalogue.add(Book("Pragmatic Programmer", "B003"))

# for loop uses __iter__ and __next__ automatically
for book in catalogue:
    print(book)

# Can also use with list(), next(), etc.
print(list(catalogue))

it = iter(catalogue)
print(next(it))      # Book('Clean Code')
print(next(it))      # Book('SICP')

Book('Clean Code')
Book('SICP')
Book('Pragmatic Programmer')
[Book('Clean Code'), Book('SICP'), Book('Pragmatic Programmer')]
Book('Clean Code')
Book('SICP')


---
## 13. Generators

In [13]:
# Generators produce values one at a time using 'yield'.
# Memory efficient — they don't build the entire result in memory.
# A function with yield IS a generator function.

class Book:
    def __init__(self, title, item_id, is_available=True):
        self.title = title
        self.item_id = item_id
        self.is_available = is_available
        self.genre = None


books = [
    Book("Clean Code", "B001", True),
    Book("SICP", "B002", False),
    Book("Pragmatic Programmer", "B003", True),
    Book("Design Patterns", "B004", False),
    Book("Refactoring", "B005", True),
]


# Generator function — yields one item at a time
def available_books(items):
    for item in items:
        if item.is_available:
            yield item           # pauses here, resumes on next call


def search_books(items, keyword):
    keyword = keyword.lower()
    for item in items:
        if keyword in item.title.lower():
            yield item


def paginate(items, page_size=2):
    """Yields chunks of items — useful for paginated display."""
    page = []
    for item in items:
        page.append(item)
        if len(page) == page_size:
            yield page
            page = []
    if page:                     # yield remaining items
        yield page


# Using generators
print("=== Available Books ===")
for book in available_books(books):
    print(f"  {book.title}")

print("\n=== Search: 'code' ===")
for book in search_books(books, "code"):
    print(f"  {book.title}")

print("\n=== Paginated (2 per page) ===")
for i, page in enumerate(paginate(books, 2), 1):
    print(f"Page {i}: {[b.title for b in page]}")

# Generator expression — like list comprehension but lazy
available_titles = (b.title for b in books if b.is_available)
print("\n=== Generator Expression ===")
print(list(available_titles))

=== Available Books ===
  Clean Code
  Pragmatic Programmer
  Refactoring

=== Search: 'code' ===
  Clean Code

=== Paginated (2 per page) ===
Page 1: ['Clean Code', 'SICP']
Page 2: ['Pragmatic Programmer', 'Design Patterns']
Page 3: ['Refactoring']

=== Generator Expression ===
['Clean Code', 'Pragmatic Programmer', 'Refactoring']


---
## 14. Decorators

In [14]:
# A decorator wraps a function to add behaviour before/after it runs.
# Syntax: @decorator above a function definition.
# Under the hood: function = decorator(function)

import time
from functools import wraps

# --- 1. Simple function decorator ---
def timer(func):
    @wraps(func)                 # preserves original function metadata
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)     # call original function
        elapsed = time.time() - start
        print(f"[TIMER] {func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper


# --- 2. Decorator with arguments (decorator factory) ---
def require_role(role):
    def decorator(func):
        @wraps(func)
        def wrapper(self, *args, **kwargs):
            if getattr(self, 'role', None) != role:
                raise PermissionError(f"Action requires '{role}' role")
            return func(self, *args, **kwargs)
        return wrapper
    return decorator


# --- 3. Logging decorator ---
def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"[LOG] {func.__name__} completed")
        return result
    return wrapper


# --- Library using decorators ---
class LibrarySystem:
    def __init__(self, role="member"):
        self.role = role
        self.books = []

    @log_action
    @timer
    def load_catalogue(self):
        """Stacking decorators — log_action wraps timer which wraps the function."""
        time.sleep(0.01)   # simulate loading
        self.books = ["Clean Code", "SICP", "Pragmatic Programmer"]
        return self.books

    @require_role("admin")
    def delete_member(self, member_id):
        print(f"Deleted member {member_id}")

    @require_role("admin")
    def view_all_fines(self):
        print("All fines: ...")


# Regular member
system = LibrarySystem(role="member")
system.load_catalogue()

try:
    system.delete_member("M001")     # blocked — not admin
except PermissionError as e:
    print(f"Blocked: {e}")

# Admin user
admin_system = LibrarySystem(role="admin")
admin_system.delete_member("M001")   # allowed

[LOG] Calling load_catalogue
[TIMER] load_catalogue took 0.0204s
[LOG] load_catalogue completed
Blocked: Action requires 'admin' role
Deleted member M001


---
## 15. Context Managers

In [15]:
# Context managers guarantee setup and teardown, even if errors occur.
# Used with the 'with' statement.
# Requires: __enter__ (setup) and __exit__ (teardown)

import json
import os
from datetime import datetime

# --- Class-based context manager ---
class LibrarySession:
    """Manages loading and saving library data around a session."""

    def __init__(self, filepath="library_data.json"):
        self.filepath = filepath
        self.data = None

    def __enter__(self):
        """Runs when 'with' block starts — load data."""
        print("[SESSION] Opening library session...")
        try:
            with open(self.filepath, 'r') as f:
                self.data = json.load(f)
            print(f"[SESSION] Loaded {len(self.data.get('books', []))} books")
        except FileNotFoundError:
            self.data = {"books": [], "members": [], "transactions": []}
            print("[SESSION] No existing data — starting fresh")
        return self.data     # this becomes the 'as' variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Runs when 'with' block ends — always, even if error occurred."""
        print("[SESSION] Saving data...")
        with open(self.filepath, 'w') as f:
            json.dump(self.data, f, indent=2)
        print("[SESSION] Session closed.")

        # Return False = don't suppress exceptions
        # Return True  = suppress exceptions (rarely correct)
        return False


# --- Generator-based context manager (alternative) ---
from contextlib import contextmanager

@contextmanager
def timed_operation(operation_name):
    """Times any block of code."""
    print(f"[TIMER] Starting: {operation_name}")
    start = time.time()
    try:
        yield                         # code in 'with' block runs here
    finally:
        elapsed = time.time() - start
        print(f"[TIMER] {operation_name} took {elapsed:.4f}s")


# --- Usage ---
with LibrarySession("library_data.json") as data:
    data["books"].append({"title": "Clean Code", "isbn": "978-01"})
    data["books"].append({"title": "SICP", "isbn": "978-02"})
    print(f"Added 2 books. Total: {len(data['books'])}")
# __exit__ runs here automatically — data is saved

print()

# Re-open and verify
with LibrarySession("library_data.json") as data:
    print(f"Reloaded. Total books: {len(data['books'])}")

print()

with timed_operation("Catalogue Search"):
    time.sleep(0.05)     # simulate work
    print("Search complete.")

# Cleanup
if os.path.exists("library_data.json"):
    os.remove("library_data.json")

[SESSION] Opening library session...
[SESSION] No existing data — starting fresh
Added 2 books. Total: 2
[SESSION] Saving data...
[SESSION] Session closed.

[SESSION] Opening library session...
[SESSION] Loaded 2 books
Reloaded. Total books: 2
[SESSION] Saving data...
[SESSION] Session closed.

[TIMER] Starting: Catalogue Search
Search complete.
[TIMER] Catalogue Search took 0.0516s


---
## 16. Everything Together — Mini Integration

In [16]:
# A condensed demo that uses ALL concepts together.
# This is the skeleton of what your full Library Management System will look like.

from abc import ABC, abstractmethod
from functools import wraps
from contextlib import contextmanager
import time

# Custom Exceptions
class LibraryError(Exception): pass
class ItemNotFoundError(LibraryError):
    def __init__(self, id): super().__init__(f"Item '{id}' not found")
class ItemNotAvailableError(LibraryError):
    def __init__(self, title): super().__init__(f"'{title}' not available")
class MemberLimitError(LibraryError):
    def __init__(self, name): super().__init__(f"{name} hit borrow limit")

# Decorator
def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"  → {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

# Mixin
class SearchMixin:
    def search(self, keyword):
        return [i for i in self.items if keyword.lower() in i.title.lower()]

# ABC
class LibraryItem(ABC):
    def __init__(self, title, item_id):
        self.title = title
        self.item_id = item_id
        self.is_available = True

    @abstractmethod
    def get_info(self) -> str: pass

    @abstractmethod
    def late_fee(self, days) -> float: pass

    def __str__(self): return self.get_info()
    def __repr__(self): return f"{self.__class__.__name__}({self.title!r})"
    def __eq__(self, other): return isinstance(other, LibraryItem) and self.item_id == other.item_id

# Concrete classes — Polymorphism
class Book(LibraryItem):
    def __init__(self, title, item_id, author):
        super().__init__(title, item_id)
        self.author = author

    def get_info(self): return f"📖 {self.title} — {self.author}"
    def late_fee(self, days): return days * 10.0

class Magazine(LibraryItem):
    def __init__(self, title, item_id, issue):
        super().__init__(title, item_id)
        self.issue = issue

    def get_info(self): return f"📰 {self.title} Issue #{self.issue}"
    def late_fee(self, days): return days * 5.0

# Library — Operator Overloading, Iterator, Mixin
class Library(SearchMixin):
    def __init__(self, name):
        self.name = name
        self.items = []

    def __add__(self, item):           # Operator overloading
        self.items.append(item)
        return self

    def __len__(self): return len(self.items)
    def __contains__(self, item): return item in self.items

    def __iter__(self):                # Iterator
        self._i = 0
        return self

    def __next__(self):
        if self._i >= len(self.items): raise StopIteration
        item = self.items[self._i]; self._i += 1; return item

    def available(self):               # Generator
        for item in self.items:
            if item.is_available:
                yield item

    @log_action                        # Decorator
    def borrow(self, item_id, member):
        item = next((i for i in self.items if i.item_id == item_id), None)
        if not item: raise ItemNotFoundError(item_id)
        if not item.is_available: raise ItemNotAvailableError(item.title)
        if len(member.borrowed) >= 3: raise MemberLimitError(member.name)
        item.is_available = False
        member.borrowed.append(item)
        print(f"     {member.name} borrowed '{item.title}'")

# Member — Encapsulation + Property
class Member:
    def __init__(self, name):
        self.name = name
        self.borrowed = []
        self.__fine = 0.0

    @property
    def fine(self): return self.__fine

    def add_fine(self, amount):
        if amount < 0: raise ValueError("Fine cannot be negative")
        self.__fine += amount

# Context Manager
@contextmanager
def library_session(library):
    print(f"\n=== Opening session: {library.name} ===")
    try:
        yield library
    finally:
        print(f"=== Session closed. {len(library)} items in catalogue ===")


# --- Run it ---
lib = Library("City Library")
lib + Book("Clean Code", "B001", "Martin")
lib + Book("SICP", "B002", "Abelson")
lib + Magazine("Time", "M001", 45)

hasnain = Member("Hasnain")

with library_session(lib) as l:
    print("\nAll items (Iterator):")
    for item in l:
        print(f"  {item}")

    print("\nBorrow:")
    try:
        l.borrow("B001", hasnain)
        l.borrow("B001", hasnain)    # will fail
    except ItemNotAvailableError as e:
        print(f"  Error: {e}")

    print("\nAvailable (Generator):")
    for item in l.available():
        print(f"  {item}")

    print("\nSearch 'code' (Mixin):")
    for item in l.search("code"):
        print(f"  {item}")

    hasnain.add_fine(50)
    print(f"\nFine (Property): Rs. {hasnain.fine}")


=== Opening session: City Library ===

All items (Iterator):
  📖 Clean Code — Martin
  📖 SICP — Abelson
  📰 Time Issue #45

Borrow:
  → borrow
     Hasnain borrowed 'Clean Code'
  → borrow
  Error: 'Clean Code' not available

Available (Generator):
  📖 SICP — Abelson
  📰 Time Issue #45

Search 'code' (Mixin):
  📖 Clean Code — Martin

Fine (Property): Rs. 50.0
=== Session closed. 3 items in catalogue ===
